# Reporting Cell Ontology term usage in CELLxGENE Census with `term_usage()`

`term_usage()` reports how a Cell Ontology (CL) term is used in [CELLxGENE Census](https://chanzuckerberg.github.io/cellxgene-census/): how many cells are annotated with it, in which tissues, developmental stages and diseases, and in which datasets and papers.

It is meant to support CL curation decisions, such as obsoleting a term or changing its name or definition in a way that broadens or narrows its meaning. Unlike `enhance()`, which expands a term to all its subtypes, `term_usage()` counts **direct annotations**: cells annotated with exactly that term.

**Notes**
- The outputs below were produced with Census release **2025-11-17** (pinned in `CENSUS_VERSION`). Numbers will differ on other releases.
- Cell counts come back in about a second. The tissue/stage context scans the cells of each organism using the term: about 20s per organism for a common term like astrocyte. Running the whole notebook takes a few minutes.

## Getting started

In [1]:
import json
import logging

import cellxgene_census
import pandas as pd
from IPython.display import Markdown

from cxg_query_enhancer import term_usage

CENSUS_VERSION = "2025-11-17"  # pin a release so results are reproducible; "latest" also works

logging.getLogger("cxg_query_enhancer").setLevel(logging.ERROR)  # hide progress messages
pd.set_option("display.max_colwidth", 80)

## 1. Quick look at a term

The term can be given as a label, an exact synonym, or a CL ID (`"CL:0000127"`). `to_markdown()` gives a short summary: cell counts per organism, the top contexts and the top papers.

In [2]:
report = term_usage("astrocyte", census_version=CENSUS_VERSION)
Markdown(report.to_markdown())

## CL:0000127 astrocyte

_A class of large neuroglial (macroglial) cells in the central nervous system - the largest and most numerous neuroglial cells in the brain and spinal cord. Astrocytes (from 'star' cells) are irregularly shaped with many long processes, including those with 'end feet' which form the glial (limiting) membrane and directly and indirectly contribute to the blood-brain barrier. They regulate the extracellular ionic and chemical environment, and 'reactive astrocytes' (along with microglia) respond to injury._

CELLxGENE Census release 2025-11-17.

**4,303,261 cells** (2,826,367 primary) in 209 datasets from 57 collections.

### Cell counts
| organism | term_id | label | total_cells | primary_cells |
|---|---|---|---|---|
| callithrix_jacchus | CL:0000127 | astrocyte | 234,817 | 165,907 |
| homo_sapiens | CL:0000127 | astrocyte | 3,218,434 | 2,266,869 |
| macaca_mulatta | CL:0000127 | astrocyte | 363,057 | 147,404 |
| mus_musculus | CL:0000127 | astrocyte | 472,276 | 231,510 |
| pan_troglodytes | CL:0000127 | astrocyte | 14,677 | 14,677 |

### Top 10 contexts
| organism | tissue | tissue_type | development_stage | disease | total_cells | primary_cells | n_datasets |
|---|---|---|---|---|---|---|---|
| mus_musculus | embryo | tissue | Theiler stage 27 | normal | 164,391 | 54,797 | 6 |
| homo_sapiens | cerebellum | tissue | 17th week post-fertilization stage | normal | 138,926 | 111,527 | 3 |
| homo_sapiens | cerebellum | tissue | 15th week post-fertilization stage | normal | 95,537 | 76,749 | 2 |
| homo_sapiens | cerebral cortex | tissue | 50-year-old stage | normal | 70,800 | 22,805 | 36 |
| homo_sapiens | dorsolateral prefrontal cortex | tissue | 80 year-old and over stage | dementia \|\| Alzheimer disease | 67,925 | 67,925 | 2 |
| macaca_mulatta | prefrontal cortex | tissue | prime adult stage | normal | 59,418 | 23,066 | 3 |
| homo_sapiens | hippocampal formation | tissue | 50-year-old stage | normal | 49,287 | 16,429 | 9 |
| callithrix_jacchus | prefrontal cortex | tissue | prime adult stage | normal | 48,811 | 25,099 | 2 |
| homo_sapiens | cerebellum | tissue | 16th week post-fertilization stage | normal | 48,345 | 38,780 | 2 |
| mus_musculus | hypothalamus | tissue | 2-month-old stage | normal | 45,432 | 45,432 | 1 |

### Top 10 papers
| collection_name | collection_doi | n_datasets | total_cells | primary_cells |
|---|---|---|---|---|
| Population-scale cross-disorder atlas of the human prefrontal cortex at single-cell resolution | 10.1101/2024.10.31.24316513 | 4 | 913,354 | 763,034 |
| Single cell atlas of the human optic nerve | (no DOI) | 2 | 511,742 | 222,473 |
| Human Brain Cell Atlas v1.0 | 10.1126/science.add7046 | 107 | 465,075 | 152,854 |
| A human cell atlas of fetal gene expression | 10.1126/science.aba7721 | 2 | 422,702 | 339,213 |
| A single-cell transcriptomic atlas spanning the adult rhesus macaque brain | 10.1126/sciadv.adh1914 | 3 | 352,124 | 136,471 |
| A single-cell transcriptional timelapse of mouse embryonic development, from gastrula to pup | 10.1038/s41586-024-07069-w | 6 | 329,250 | 109,750 |
| A multi-region single nucleus transcriptomic atlas of Parkinson’s disease | 10.1038/s41597-024-04117-y | 1 | 227,452 | 227,452 |
| A marmoset brain cell census reveals influence of developmental origin and functional class on neuronal identity | 10.1126/sciadv.adk3986 | 2 | 217,871 | 149,761 |
| An integrated transcriptomic cell atlas of human neural organoids | 10.1038/s41586-024-08172-8 | 2 | 138,673 | 71,683 |
| Single-cell analysis of prenatal and postnatal human cortical development | 10.1126/science.adf0834 | 1 | 67,868 | 67,868 |

## 2. Exploring the tables

The report holds four pandas DataFrames:

| Table | Content |
|---|---|
| `cell_counts` | cells per organism (and per term) |
| `context` | cells per organism × tissue × tissue type × development stage × disease, with the number of datasets |
| `datasets` | cells per dataset, with title, citation and collection (paper) |
| `papers` | cells per collection, with name and DOI |

Every table has `total_cells` and `primary_cells`. The same cells can appear in several datasets, so **`primary_cells` (unique cells) is the number to use**; `total_cells` double-counts.

In [3]:
report.cell_counts

,organism,term_id,label,total_cells,primary_cells
0,callithrix_jacchus,CL:0000127,astrocyte,234817,165907
1,homo_sapiens,CL:0000127,astrocyte,3218434,2266869
2,macaca_mulatta,CL:0000127,astrocyte,363057,147404
3,mus_musculus,CL:0000127,astrocyte,472276,231510
4,pan_troglodytes,CL:0000127,astrocyte,14677,14677


Tissues where human astrocytes are annotated, by unique cells:

In [4]:
human = report.context[report.context.organism == "homo_sapiens"]
human.groupby("tissue")["primary_cells"].sum().sort_values(ascending=False).head(10)

tissue
dorsolateral prefrontal cortex    801122
cerebellum                        300861
cranial nerve II                  179221
optic disc                         80698
prefrontal cortex                  79830
frontal cortex                     78148
cerebral cortex                    75409
primary motor cortex               62795
hypothalamus                       61710
telencephalon                      59583
Name: primary_cells, dtype: int64

Papers to read when reviewing the definition, ranked by unique cells:

In [5]:
report.papers.sort_values("primary_cells", ascending=False)[
    ["collection_name", "collection_doi", "n_datasets", "primary_cells"]
].head(10)

,collection_name,collection_doi,n_datasets,primary_cells
0,Population-scale cross-disorder atlas of the human prefrontal cortex at sing...,10.1101/2024.10.31.24316513,4,763034
3,A human cell atlas of fetal gene expression,10.1126/science.aba7721,2,339213
6,A multi-region single nucleus transcriptomic atlas of Parkinson’s disease,10.1038/s41597-024-04117-y,1,227452
1,Single cell atlas of the human optic nerve,,2,222473
2,Human Brain Cell Atlas v1.0,10.1126/science.add7046,107,152854
7,A marmoset brain cell census reveals influence of developmental origin and f...,10.1126/sciadv.adk3986,2,149761
4,A single-cell transcriptomic atlas spanning the adult rhesus macaque brain,10.1126/sciadv.adh1914,3,136471
5,"A single-cell transcriptional timelapse of mouse embryonic development, from...",10.1038/s41586-024-07069-w,6,109750
8,An integrated transcriptomic cell atlas of human neural organoids,10.1038/s41586-024-08172-8,2,71683
9,Single-cell analysis of prenatal and postnatal human cortical development,10.1126/science.adf0834,1,67868


Contexts with a disease other than `normal`, e.g. to see whether the term is used for reactive cells:

In [6]:
report.context[report.context.disease != "normal"].head(10)

,organism,tissue,tissue_type,development_stage,disease,total_cells,primary_cells,n_datasets
4,homo_sapiens,dorsolateral prefrontal cortex,tissue,80 year-old and over stage,dementia || Alzheimer disease,67925,67925,2
83,homo_sapiens,dorsolateral prefrontal cortex,tissue,80 year-old and over stage,dementia,10271,10271,1
95,homo_sapiens,dorsolateral prefrontal cortex,tissue,84-year-old stage,dementia || Alzheimer disease,9412,9412,2
98,homo_sapiens,dorsolateral prefrontal cortex,tissue,88-year-old stage,dementia || Alzheimer disease,9049,9049,2
102,homo_sapiens,dorsolateral prefrontal cortex,tissue,80 year-old and over stage,dementia || Alzheimer disease || Lewy body dementia,8726,8726,2
111,homo_sapiens,right temporal lobe,tissue,63-year-old stage,glioblastoma,8372,8372,1
115,homo_sapiens,dorsolateral prefrontal cortex,tissue,83-year-old stage,dementia || Alzheimer disease,7803,7803,2
119,homo_sapiens,dorsolateral prefrontal cortex,tissue,89-year-old stage,dementia || Alzheimer disease,7505,7505,2
121,homo_sapiens,dorsolateral prefrontal cortex,tissue,80-year-old stage,dementia || Alzheimer disease,7421,7421,2
127,homo_sapiens,dorsolateral prefrontal cortex,tissue,80 year-old and over stage,dementia || vascular dementia || Alzheimer disease,7150,7150,2


Datasets whose cells all appear in other datasets as well (no unique cells):

In [7]:
no_unique = report.datasets[report.datasets.primary_cells == 0]
print(f"{len(no_unique)} of {len(report.datasets)} datasets have no unique cells")
no_unique[["organism", "dataset_title", "collection_name", "total_cells"]].head(10)

35 of 209 datasets have no unique cells


,organism,dataset_title,collection_name,total_cells
2,homo_sapiens,snRNA-seq of human optic nerve and optic nerve head - astrocytes,Single cell atlas of the human optic nerve,255871
6,homo_sapiens,All non-neuronal cells,Human Brain Cell Atlas v1.0,155025
7,homo_sapiens,Supercluster: Astrocyte,Human Brain Cell Atlas v1.0,155025
8,homo_sapiens,Aging_Cohort,Population-scale cross-disorder atlas of the human prefrontal cortex at sing...,150320
10,macaca_mulatta,A single-cell multi-omic atlas spanning the adult rhesus macaque brain (cell...,A single-cell transcriptomic atlas spanning the adult rhesus macaque brain,136471
13,mus_musculus,Major cell cluster: Neuroectoderm and glia,"A single-cell transcriptional timelapse of mouse embryonic development, from...",109750
15,homo_sapiens,Survey of human embryonic development (1 million cells subset),A human cell atlas of fetal gene expression,83489
16,macaca_mulatta,A single-cell multi-omic atlas spanning the adult rhesus macaque brain (1.5 ...,A single-cell transcriptomic atlas spanning the adult rhesus macaque brain,79182
20,homo_sapiens,The Human Neural Organoid Atlas,An integrated transcriptomic cell atlas of human neural organoids,66990
26,mus_musculus,Whole dataset: Normalized subset 1,"A single-cell transcriptional timelapse of mouse embryonic development, from...",27478


## 3. Before obsoleting a term: where are the cells among its subtypes?

`include_descendants=True` also reports each subclass of the term. Every term is still counted by its **own** direct annotations, so the table shows how many cells sit on the term itself and how many on each subtype. Here limited to human and mouse to keep it quick.

In [8]:
report_tree = term_usage(
    "CL:0000127",
    organisms=["homo_sapiens", "mus_musculus"],
    include_descendants=True,
    census_version=CENSUS_VERSION,
)
print(f"{len(report_tree.terms)} terms in the closure, {report_tree.cell_counts.term_id.nunique()} used in Census")

report_tree.cell_counts.pivot_table(
    index=["term_id", "label"], columns="organism", values="primary_cells", aggfunc="sum", fill_value=0
).sort_values("homo_sapiens", ascending=False)

59 terms in the closure, 8 used in Census


,organism,homo_sapiens,mus_musculus
term_id,label,,
CL:0000127,astrocyte,2266869,231510
CL:0002605,astrocyte of the cerebral cortex,191414,0
CL:0002604,hippocampal astrocyte,27092,0
CL:0002627,mature astrocyte,15250,0
CL:4033015,retinal astrocyte,14954,0
CL:0000644,Bergmann glial cell,13815,17733
CL:0002626,immature astrocyte,5494,810
CL:0002603,astrocyte of the cerebellum,0,16717


## 4. Several terms at once

Reusing one open Census connection avoids reopening it for every term. When `census=` is given, the release is taken from that connection.

In [9]:
terms = ["astrocyte", "oligodendrocyte", "microglial cell"]

with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    reports = {t: term_usage(t, organisms=["homo_sapiens"], census=census) for t in terms}

pd.DataFrame(
    [
        {
            "term": t,
            "term_id": r.term_id,
            "primary_cells": int(r.cell_counts.primary_cells.sum()),
            "datasets": len(r.datasets),
            "papers": len(r.papers),
            "tissues": r.context.tissue.nunique(),
        }
        for t, r in reports.items()
    ]
)

,term,term_id,primary_cells,datasets,papers,tissues
0,astrocyte,CL:0000127,2266869,173,44,80
1,oligodendrocyte,CL:0000128,5777382,192,44,90
2,microglial cell,CL:0000129,920129,89,52,97


## 5. Exporting for other tools

`to_dict()` returns plain dicts and lists that can be saved as JSON, e.g. as input to a workflow that reads the papers. Each dataset's `citation` contains the publication DOI and the `.h5ad` download link. Collections without a DOI have `collection_doi == ""`.

In [10]:
data = report.to_dict()
print({key: len(value) if isinstance(value, list) else value for key, value in data.items() if key != "definition"})
print(json.dumps(data["papers"][0], indent=1))

# with open("astrocyte_usage.json", "w") as f:
#     json.dump(data, f, indent=1)

{'term_id': 'CL:0000127', 'label': 'astrocyte', 'census_version': '2025-11-17', 'terms': 1, 'cell_counts': 5, 'context': 1727, 'datasets': 209, 'papers': 57}
{
 "collection_id": "84ce6837-548d-4a1f-919f-0bc0d9a3952f",
 "collection_name": "Population-scale cross-disorder atlas of the human prefrontal cortex at single-cell resolution",
 "collection_doi": "10.1101/2024.10.31.24316513",
 "n_datasets": 4,
 "total_cells": 913354,
 "primary_cells": 763034
}
